In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
DATA_DIR = path if "path" in globals() and os.path.exists(f"{path}/train_features.csv") else "data"
CACHE_DIR = "rf/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
_NAMES = ("X_train", "y_train", "X_test", "test_ids")

def load_data():
  paths = [f"{CACHE_DIR}/{n}.npy" for n in _NAMES]
  if all(os.path.exists(p) for p in paths):
    return [np.load(p) for p in paths]
  feats = [c for c in pd.read_csv(f"{DATA_DIR}/train_features.csv", nrows=0).columns
           if c not in ("id", "label")]
  dtype = {c: np.float32 for c in feats} | {"id": str, "label": np.int8}
  tr = pd.read_csv(f"{DATA_DIR}/train_features.csv", dtype=dtype)
  te = pd.read_csv(f"{DATA_DIR}/test_features.csv", dtype=dtype)
  assert feats == [c for c in te.columns if c != "id"], "train/test feature columns differ"
  arrs = [tr[feats].to_numpy(np.float32), tr["label"].to_numpy(np.int8),
          te[feats].to_numpy(np.float32), np.asarray(te["id"], dtype=np.str_)]
  del tr, te
  for p, a in zip(paths, arrs):
    np.save(p, a)
  return arrs

X_train, y_train, X_test, test_ids = load_data()

assert X_train.shape[1] == X_test.shape[1] == 5000
assert len(X_train) == len(y_train) and len(X_test) == len(test_ids)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
assert set(np.unique(y_train)) == {0, 1}
print(X_train.shape, X_test.shape, X_train.dtype,
      f"| {X_train.nbytes / 2**20:.0f} MB train | machine frac {y_train.mean():.3f}")

(20000, 5000) (6999, 5000) float32 | 381 MB train | machine frac 0.625


In [4]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

RESULTS_CSV = "rf/results.csv"
THRESHOLDS = np.arange(0.30, 0.9001, 0.01)
BASE = dict(
    n_estimators=300,          # fixed across stage 1 so axes are comparable
    max_features='sqrt',       # ~70 of 5000 columns per split
    min_samples_leaf=1,
    min_samples_split=2,
    max_depth=None,
    criterion='gini',
    class_weight=None,
    bootstrap=True,            # required for OOB scoring
    n_jobs=8,
    random_state=42,
    verbose=0,
)

def macro_f1(y_true, y_pred):
  return f1_score(y_true, y_pred, average='macro')

def make_rf(**kwargs):
  return RandomForestClassifier(**{**BASE, 'oob_score': macro_f1, **kwargs})

def best_threshold(y, proba):
  scores = np.array([macro_f1(y, (proba >= t).astype(np.int8)) for t in THRESHOLDS])
  i = int(scores.argmax())
  return float(THRESHOLDS[i]), float(scores[i])

def score_oob(m, y):
  """Tuned macro F1 from the out-of-bag decision function."""
  proba = m.oob_decision_function_[:, 1]
  ok = ~np.isnan(proba)                      # rows that were never out-of-bag
  return best_threshold(y[ok], proba[ok])

RESULTS = []

def record(rec):
  RESULTS.append(rec)
  pd.DataFrame(RESULTS).to_csv(RESULTS_CSV, index=False)   # rewritten after every fit

def eval_model(axis=None, X=None, y=None, **kwargs):
  X = X_train if X is None else X
  y = y_train if y is None else y
  t0 = time.perf_counter()
  m = make_rf(**kwargs).fit(X, y)
  thr, f1_tuned = score_oob(m, y)
  rec = dict(axis=axis, **kwargs,
              oob_f1=float(m.oob_score_),     # at the implicit 0.5 cutoff
              oob_f1_tuned=f1_tuned, threshold=thr,
              nodes=int(sum(t.tree_.node_count for t in m.estimators_)),
              fit_s=round(time.perf_counter() - t0, 1))
  del m
  record(rec)
  print(f"{rec['fit_s']:>6.1f}s  oob={rec['oob_f1']:.4f}  tuned={f1_tuned:.4f}"
        f"  thr={thr:.2f}  {axis}={kwargs.get(axis)}")
  return rec

In [5]:
SCREEN = dict(n_estimators=100)
PARAM_GRID = {
  'min_samples_leaf': [2, 5, 10, 20],
  'criterion':        ['entropy'],
}

base = [eval_model(axis='random_state', random_state=s, **SCREEN) for s in (42, 0, 1, 2, 3)]
noise = float(np.std([r['oob_f1_tuned'] for r in base]))
print(f"\nseed noise sigma @100 trees = {noise:.4f} -> treat any axis spread under {2*noise:.4f} as flat\n")

for param, values in PARAM_GRID.items():
  for v in values:
    eval_model(axis=param, **SCREEN, **{param: v})

  56.3s  oob=0.7074  tuned=0.7317  thr=0.59  random_state=42
  47.3s  oob=0.7001  tuned=0.7295  thr=0.59  random_state=0
  60.4s  oob=0.6996  tuned=0.7294  thr=0.58  random_state=1
  65.9s  oob=0.7054  tuned=0.7326  thr=0.59  random_state=2
  62.4s  oob=0.6997  tuned=0.7286  thr=0.59  random_state=3

seed noise sigma @100 trees = 0.0015 -> treat any axis spread under 0.0030 as flat

  56.1s  oob=0.6845  tuned=0.7272  thr=0.62  min_samples_leaf=2
  57.4s  oob=0.6776  tuned=0.7281  thr=0.59  min_samples_leaf=5
  52.0s  oob=0.6592  tuned=0.7214  thr=0.60  min_samples_leaf=10
  54.6s  oob=0.6341  tuned=0.7123  thr=0.60  min_samples_leaf=20
  68.4s  oob=0.7041  tuned=0.7318  thr=0.60  criterion=entropy


In [ ]:
curve = make_rf(n_estimators=100, warm_start=True)
for n in (100, 200, 400, 800, 1200):
  curve.set_params(n_estimators=n)
  t0 = time.perf_counter()
  curve.fit(X_train, y_train)
  thr, f1_tuned = score_oob(curve, y_train)
  record(dict(axis='n_estimators', n_estimators=n,
              oob_f1=float(curve.oob_score_), oob_f1_tuned=f1_tuned, threshold=thr,
              nodes=int(sum(t.tree_.node_count for t in curve.estimators_)),
              fit_s=round(time.perf_counter() - t0, 1)))
  print(f"{n:>5} trees  oob={curve.oob_score_:.4f}  tuned={f1_tuned:.4f}  thr={thr:.2f}"
        f"  (+{RESULTS[-1]['fit_s']:.0f}s)")
del curve

  100 trees  oob=0.7074  tuned=0.7317  thr=0.59  (+74s)


In [ ]:
df = pd.DataFrame(RESULTS)
df.to_csv('rf/results.csv', index=False)

for param in PARAM_GRID:
  sub = df[df.axis == param]
  if len(sub):
    spread = sub.oob_f1_tuned.max() - sub.oob_f1_tuned.min()
    print(f"{param:<18} spread={spread:.4f}  {'MATTERS' if spread > 2*noise else 'flat'}")
df.sort_values('oob_f1_tuned', ascending=False).head(10)

min_samples_leaf   spread=0.0158  MATTERS
criterion          spread=0.0000  flat


,axis,random_state,n_estimators,oob_f1,oob_f1_tuned,threshold,nodes,fit_s,min_samples_leaf,criterion
14,n_estimators,NaN,1200,0.705263,0.756084,0.61,8952922,528.0,NaN,NaN
13,n_estimators,NaN,800,0.703305,0.753054,0.58,5965108,394.9,NaN,NaN
12,n_estimators,NaN,400,0.705899,0.748950,0.61,2989088,194.7,NaN,NaN
11,n_estimators,NaN,200,0.709129,0.742461,0.58,1492294,92.1,NaN,NaN
3,random_state,2.0,100,0.705380,0.732562,0.59,749116,56.2,NaN,NaN
9,criterion,NaN,100,0.704113,0.731849,0.60,722182,56.7,NaN,entropy
0,random_state,42.0,100,0.707365,0.731677,0.59,745284,53.5,NaN,NaN
10,n_estimators,NaN,100,0.707365,0.731677,0.59,745284,60.2,NaN,NaN
1,random_state,0.0,100,0.700074,0.729543,0.59,746816,47.5,NaN,NaN
2,random_state,1.0,100,0.699582,0.729359,0.58,743618,51.5,NaN,NaN


In [ ]:
pd.DataFrame(RESULTS).to_csv('rf_results.csv', index=False)

In [ ]:
FINAL = dict(n_estimators=1200)

t0 = time.perf_counter()
final = make_rf(**FINAL).fit(X_train, y_train)
THR, oob_tuned = score_oob(final, y_train)
print(f"{time.perf_counter() - t0:.0f}s  oob={final.oob_score_:.4f}  tuned={oob_tuned:.4f}  thr={THR:.2f}")

proba_test = final.predict_proba(X_test)[:, 1]
print("\nthreshold sensitivity (machine fraction on test):")
for t in (0.50, 0.55, 0.58, 0.59, 0.60, 0.61, 0.65):
  print(f"  thr={t:.2f} -> {(proba_test >= t).mean():.3f}")

pred = (proba_test >= THR).astype(np.int8)
sub = pd.DataFrame({"id": test_ids, "label": pred})
assert len(sub) == 6999 and set(np.unique(sub.label)) <= {0, 1}
sub.to_csv("rf/RandomForest_Prediction.csv", index=False)
print(f"\nwrote rf/RandomForest_Prediction.csv at thr={THR:.2f}"
      f" | machine frac {pred.mean():.3f} vs train {y_train.mean():.3f}")
sub.head()

760s  oob=0.7053  tuned=0.7561  thr=0.61

threshold sensitivity (machine fraction on test):
  thr=0.50 -> 0.884
  thr=0.55 -> 0.828
  thr=0.58 -> 0.784
  thr=0.59 -> 0.767
  thr=0.60 -> 0.750
  thr=0.61 -> 0.731
  thr=0.65 -> 0.643

wrote rf/RandomForest_Prediction.csv at thr=0.61 | machine frac 0.729 vs train 0.625


,id,label
0,59218,1
1,37110,1
2,23200,1
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,1


In [ ]:
PI_TEST = 0.54

r = (PI_TEST / (1 - PI_TEST)) / (y_train.mean() / (1 - y_train.mean()))
THR_PRIOR = float(THR / (THR + r * (1 - THR)))
THR_BAL = float(np.quantile(proba_test, 1 - PI_TEST))

for name, t in (("oob", THR), ("prior", THR_PRIOR), ("balanced", THR_BAL)):
  print(f"{name:<9} thr={t:.3f} -> machine frac {(proba_test >= t).mean():.3f}")

for name, t in (("prior", THR_PRIOR), ("balanced", THR_BAL)):
  p = (proba_test >= t).astype(np.int8)
  out = f"rf/RandomForest_Prediction_{name}.csv"
  pd.DataFrame({"id": test_ids, "label": p}).to_csv(out, index=False)
  print(f"wrote {out}")

oob       thr=0.610 -> machine frac 0.729
prior     thr=0.690 -> machine frac 0.550
balanced  thr=0.693 -> machine frac 0.542
wrote rf/RandomForest_Prediction_prior.csv
wrote rf/RandomForest_Prediction_balanced.csv


In [ ]:
def em_prior(proba, pi_tr, iters=200, tol=1e-8):
  """Saerens et al. (2002) EM estimate of the test-set class prior."""
  pi = pi_tr
  for _ in range(iters):
    w = (pi / pi_tr) * proba
    adj = w / (w + ((1 - pi) / (1 - pi_tr)) * (1 - proba))
    pi_new = float(adj.mean())
    if abs(pi_new - pi) < tol:
      break
    pi = pi_new
  return pi

PI_TR = float(y_train.mean())
pi_em = em_prior(proba_test, PI_TR)
print(f"train prior              {PI_TR:.3f}")
print(f"mean proba on test       {proba_test.mean():.3f}")
print(f"EM estimate (unlabelled) {pi_em:.3f}")
print(f"leaderboard estimate     {PI_TEST:.3f}")


train prior              0.625
mean proba on test       0.698
EM estimate (unlabelled) 0.983
leaderboard estimate     0.540


In [ ]:
from sklearn.metrics import confusion_matrix
_oob_p = final.oob_decision_function_[:, 1]
_ok = ~np.isnan(_oob_p)
tn, fp, fn, tp = confusion_matrix(y_train[_ok], (_oob_p[_ok] >= THR).astype(np.int8)).ravel()
TPR, FPR = tp / (tp + fn), fp / (fp + tn)
rate_test = float((proba_test >= THR).mean())
pi_acc = (rate_test - FPR) / (TPR - FPR)

print(f"OOB at thr={THR:.2f}:  TPR={TPR:.3f}  FPR={FPR:.3f}")
print(f"predicted machine rate on test = {rate_test:.3f}")
print(f"ACC prior estimate = {pi_acc:.3f}   (leaderboard says {PI_TEST:.3f})")
implied_fpr = (rate_test - PI_TEST * TPR) / (1 - PI_TEST)
print(f"\nimplied test FPR if pi=0.540 and TPR held: {implied_fpr:.3f}"
      f"  (vs {FPR:.3f} out of bag)")

OOB at thr=0.61:  TPR=0.775  FPR=0.249
predicted machine rate on test = 0.729
ACC prior estimate = 0.913   (leaderboard says 0.540)

implied test FPR if pi=0.540 and TPR held: 0.675  (vs 0.249 out of bag)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
oob_p = final.oob_decision_function_[:, 1]
ok = ~np.isnan(oob_p)
oob_pred = (oob_p[ok] >= THR).astype(np.int8)
print(confusion_matrix(y_train[ok], oob_pred), "\n")
print(classification_report(y_train[ok], oob_pred, target_names=["human", "machine"], digits=3))
print(f"train macro F1 = {macro_f1(y_train, final.predict(X_train)):.4f}")
imp = np.sort(final.feature_importances_)[::-1]
print(f"impurity importance: top 50 = {imp[:50].sum():.1%}, top 500 = {imp[:500].sum():.1%}")
nz = (X_train > 0).sum(1)
print(f"nonzero features per doc: median {np.median(nz):.0f}, mean {nz.mean():.1f} of 5000")

[[5630 1866]
 [2811 9693]] 

              precision    recall  f1-score   support

       human      0.667     0.751     0.707      7496
     machine      0.839     0.775     0.806     12504

    accuracy                          0.766     20000
   macro avg      0.753     0.763     0.756     20000
weighted avg      0.774     0.766     0.768     20000

train macro F1 = 1.0000
impurity importance: top 50 = 15.0%, top 500 = 47.5%
nonzero features per doc: median 58, mean 67.8 of 5000


In [ ]:
np.save("rf/cache/oob_proba_tfidf.npy", final.oob_decision_function_[:, 1])
np.save("rf/cache/proba_test_tfidf.npy", proba_test)

In [ ]:
import re
from collections import Counter
COUNT_PATTERNS = {
  "hard_wrap":  (r"[a-z,]\n[a-z]", 0),                      # arXiv mid-sentence wrap: 9.2% human vs 0.1% machine
  "ptb_clitic": (r"\s(?:n't|'s|'re|'ve|'ll|'d|'m)\b", 0),   # pre-tokenised text: 659 human docs vs 18
  "wiki":       (r"External links|References|\bURL\b", 0),
  "paren":      (r"\(", 0),
  "digit":      (r"\d", 0),
  "semicolon":  (r";", 0),
  "allcaps":    (r"\b[A-Z]{4,}\b", 0),
  "latex":      (r"\$[^$]+\$|\\[a-zA-Z]+", 0),
  "newline":    (r"\n", 0),
  "para_break": (r"\n\n", 0),
  "llm_vocab":  (r"\b(?:crucial|comprehensive|additionally|furthermore|moreover|ensuring|"
                 r"insights|implications|valuable|diverse|challenges|significantly|significant|"
                 r"potential|understanding|providing|benefits)\b", re.I),
  "conclusion": (r"in conclusion|in summary|overall,|it is important to", re.I),
  "hedging":    (r"\b(?:basically|mostly|probably|usually|pretty|quite|couple|generally|bit)\b", re.I),
  "however":    (r"however,", re.I),
}
SENT_SPLIT = re.compile(r"[.!?]+[\s\n]")
WORD = re.compile(r"[a-z']+")

def structural_features(texts):
  """40 dense columns: 4 length stats, 14 regex counts, the same 14 per 1000 words,
  and 8 stylometry measures. Counts are uncapped -- a tree can split anywhere."""
  comp = {k: re.compile(p, f) for k, (p, f) in COUNT_PATTERNS.items()}
  rows = []
  for t in texts:
    t = t if isinstance(t, str) else ""
    words = t.split()
    n_words = len(words)
    d = max(n_words, 1)
    counts = [len(c.findall(t)) for c in comp.values()]

    wl = np.array([len(w) for w in words], dtype=np.float32) if n_words else np.zeros(1, np.float32)
    sents = [s for s in SENT_SPLIT.split(t) if s.strip()]
    sl = np.array([len(s.split()) for s in sents], dtype=np.float32) if sents else np.zeros(1, np.float32)
    toks = WORD.findall(t.lower())
    n_toks = max(len(toks), 1)
    freq = Counter(toks)
    hapax = sum(1 for v in freq.values() if v == 1)
    bg = list(zip(toks, toks[1:]))
    rep_bigram = 1.0 - len(set(bg)) / len(bg) if bg else 0.0

    rows.append(
      [len(t), n_words, float(wl.mean()), len(t) / d]
      + counts
      + [1000.0 * c / d for c in counts]
      + [len(sents), float(sl.mean()), float(sl.std()), float(sl.max()),
         float(wl.std()), len(freq) / n_toks, hapax / n_toks, rep_bigram]
    )
  names = (["n_chars", "n_words", "avg_word_len", "chars_per_word"]
           + list(comp) + [f"{k}_per_kw" for k in comp]
           + ["n_sentences", "mean_sent_len", "std_sent_len", "max_sent_len",
              "std_word_len", "type_token_ratio", "hapax_ratio", "repeated_bigram_rate"])
  return np.asarray(rows, dtype=np.float32), names
if all(os.path.exists(f"{CACHE_DIR}/{n}.npy") for n in ("S_train", "S_test")):
  S_train, S_test = np.load(f"{CACHE_DIR}/S_train.npy"), np.load(f"{CACHE_DIR}/S_test.npy")
  struct_names = open(f"{CACHE_DIR}/S_names.txt").read().split()
else:
  _tr = pd.read_csv(f"{DATA_DIR}/train.csv", dtype={"id": str})
  _te = pd.read_csv(f"{DATA_DIR}/test.csv", dtype={"id": str})
  S_train, struct_names = structural_features(_tr["text"].tolist())
  S_test, _ = structural_features(_te["text"].tolist())
  del _tr, _te
  np.save(f"{CACHE_DIR}/S_train.npy", S_train)
  np.save(f"{CACHE_DIR}/S_test.npy", S_test)
  open(f"{CACHE_DIR}/S_names.txt", "w").write("\n".join(struct_names))

print(S_train.shape, S_test.shape)
print(f"{'feature':<22}{'human':>10}{'machine':>10}")
for _n in ("hard_wrap", "ptb_clitic", "llm_vocab", "repeated_bigram_rate", "hapax_ratio"):
  _j = struct_names.index(_n)
  print(f"{_n:<22}{S_train[y_train == 0, _j].mean():10.3f}{S_train[y_train == 1, _j].mean():10.3f}")

In [ ]:
XS_train = np.hstack([X_train, S_train])
XS_test = np.hstack([X_test, S_test])

t0 = time.perf_counter()
rf_struct = make_rf(n_estimators=1200).fit(XS_train, y_train)
THR_S, oob_s = score_oob(rf_struct, y_train)
print(f"{time.perf_counter() - t0:.0f}s  oob={rf_struct.oob_score_:.4f}  tuned={oob_s:.4f}  thr={THR_S:.2f}")

_p = rf_struct.oob_decision_function_[:, 1]
_ok = ~np.isnan(_p)
print(confusion_matrix(y_train[_ok], (_p[_ok] >= THR_S).astype(np.int8)), "\n")
print(classification_report(y_train[_ok], (_p[_ok] >= THR_S).astype(np.int8),
                            target_names=["human", "machine"], digits=3))
proba_test_struct = rf_struct.predict_proba(XS_test)[:, 1]
np.save(f"{CACHE_DIR}/proba_test_struct.npy", proba_test_struct)
THR_BAL_S = float(np.quantile(proba_test_struct, 1 - PI_TEST))
print(f"oob thr {THR_S:.2f} -> machine {(proba_test_struct >= THR_S).mean():.3f}"
      f" | balanced thr {THR_BAL_S:.3f} -> machine {(proba_test_struct >= THR_BAL_S).mean():.3f}")

_lab = (proba_test_struct >= THR_BAL_S).astype(np.int8)
assert len(_lab) == 6999 and set(np.unique(_lab)) <= {0, 1}
pd.DataFrame({"id": test_ids, "label": _lab}).to_csv(
    "rf/RandomForest_Prediction_struct_balanced.csv", index=False)
print("wrote rf/RandomForest_Prediction_struct_balanced.csv  (public LB 0.736)")